# Autonomee: A Config-Driven AI Agent Harness

**Series:** Agentic Harness — [Part 1](https://sriharshacr.github.io/blogs/ai-agent-scaffolding-four-failures/) | **Part 2**

This notebook runs the same three-agent pipeline two ways — **raw** (no harness protections) and **harness** (Autonomee active) — then zooms into each of the four architectural decisions.

---

Connecting Dots... 👉 https://sriharshacr.github.io/blogs/

## Setup

**What you need:**
1. A free [Groq account](https://console.groq.com) — no credit card required
2. A Groq API key added to Colab Secrets as `GROQ_API_KEY`

**To add a secret:** Click the key icon (🔑) in the left sidebar → Add new secret → Name: `GROQ_API_KEY` → Paste your key

Then run all cells top to bottom.

In [ ]:
%pip install openai --quiet

In [ ]:
# ── Provider config ───────────────────────────────────────────────────────────
# Default: Groq (free, no credit card required).
# To switch providers: update BASE_URL and the matching secret name.
#
BASE_URL = 'https://api.groq.com/openai/v1'
MODEL = 'groq/compound-mini'
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import userdata
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=userdata.get('GROQ_API_KEY'))
print(f'Client ready. Model: {MODEL}')

In [ ]:
from datetime import datetime

def ask(agent_id, prompt, system=None):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    inp = response.usage.prompt_tokens
    out = response.usage.completion_tokens
    return {
        'agent_id': agent_id,
        'output': response.choices[0].message.content,
        'input_tokens': inp,
        'output_tokens': out,
        'total_tokens': inp + out,
    }

def log(agent_id, event, detail=''):
    ts = datetime.now().strftime('%H:%M:%S')
    suffix = f' — {detail}' if detail else ''
    print(f'[{ts}] [{agent_id}] {event}{suffix}')

print('Helpers ready.')

In [ ]:
TOPIC = 'AI agent orchestration'  # ← change this to any topic

SIMULATE_FAILURE = True   # inject a failure in the writer agent to show contrast
DEMO_AUTO_APPROVE = True  # auto-approve publisher gate (set False for manual review)

CONFIG = {
    'agents': [
        {
            'id': 'researcher',
            'name': 'Research Agent',
            'system': 'Summarize this topic in exactly 3 concise bullet points.',
            'tools': ['search'],
            'input': {'source': 'config', 'key': 'topic'},
            'approval': {'required': False},
        },
        {
            'id': 'writer',
            'name': 'Writer Agent',
            'system': 'Based on the research provided, write a 60-80 word paragraph.',
            'tools': ['read'],
            'input': {'source': 'agent', 'agent_id': 'researcher'},
            'approval': {'required': False},
        },
        {
            'id': 'publisher',
            'name': 'Publisher Agent',
            'system': 'Suggest one blog post title and one specific improvement for this draft.',
            'tools': ['post'],
            'input': {'source': 'agent', 'agent_id': 'writer'},
            'approval': {'required': True},
        },
    ]
}

ALL_TOOLS = {
    'search':     lambda q: f'[search] called: {q[:40]}',
    'read':       lambda q: f'[read] called: {q[:40]}',
    'write':      lambda q: f'[write] called: {q[:40]}',
    'post':       lambda q: f'[post] called: {q[:40]}',
    'direct_llm': lambda q: f'[direct_llm] unauthorized: {q[:40]}',
}

print('Config ready. Pipeline: researcher -> writer -> publisher')
print(f'Topic: {TOPIC}')

## The Pipeline

Three agents. One topic. Two ways to run.

| Agent | Role | Allowed Tools | Input |
|---|---|---|---|
| researcher | Summarize the topic into 3 bullets | `search` | TOPIC |
| writer | Turn research into a short paragraph | `read` | researcher output |
| publisher | Suggest a title and one improvement | `post` | writer output |

In both runs, **the writer will fail** (`SIMULATE_FAILURE = True`). The difference is what happens next.

In [ ]:
def run_raw(topic):
    SEP = '=' * 62
    STALE = 'AI automates tasks. Agents follow instructions. Models generate text.'

    print(SEP)
    print('RAW PIPELINE — no harness protections')
    print(SEP)

    results = {}

    for agent_cfg in CONFIG['agents']:
        aid = agent_cfg['id']
        name = agent_cfg['name']

        # Input: stale string if agent-to-agent — no in-memory handoff
        if agent_cfg['input']['source'] == 'agent':
            agent_input = STALE
        else:
            agent_input = topic

        print()
        print(f'Running {name}...')

        # Inject failure in writer — swallowed here
        if aid == 'writer' and SIMULATE_FAILURE:
            try:
                raise RuntimeError('model timeout — writer agent failed')
            except Exception as e:
                print(f'  error: {e}')
                print('  swallowed — continuing to next agent')
                results[aid] = ''
                continue

        try:
            result = ask(aid, agent_input, agent_cfg.get('system'))
            results[aid] = result['output']
            tokens = result['total_tokens']
            print(f'  done ({tokens} tokens)')
        except Exception as e:
            print(f'  error: {e} — swallowed')
            results[aid] = ''

    print()
    print(SEP)
    print('RAW PIPELINE ENDED')
    out = results.get('publisher', '')
    print()
    if out:
        print(f'Publisher: {out[:300]}')
    else:
        print('Publisher: (empty — ran on empty writer output)')
    print(SEP)
    return results


def run_harness(topic):
    SEP = '=' * 62

    print(SEP)
    print('HARNESS PIPELINE — Autonomee protections active')
    print(SEP)

    results = {}

    for agent_cfg in CONFIG['agents']:
        aid = agent_cfg['id']
        name = agent_cfg['name']

        # Input: in-memory from prior agent — no stale reads
        if agent_cfg['input']['source'] == 'agent':
            upstream = agent_cfg['input']['agent_id']
            if upstream not in results:
                log(aid, 'INPUT_MISSING', f'no output from {upstream}')
                raise RuntimeError(f'missing input from {upstream}')
            agent_input = results[upstream]
        else:
            agent_input = topic

        # Tool scope: only config-declared tools
        allowed = set(agent_cfg['tools'])
        log(aid, 'TOOLS_REGISTERED', str(sorted(allowed)))

        # Inject failure in writer — propagates here, never swallowed
        if aid == 'writer' and SIMULATE_FAILURE:
            log(aid, 'START')
            log(aid, 'FAILURE', 'model timeout — writer agent failed')
            raise RuntimeError('writer agent failed')

        log(aid, 'START')
        try:
            result = ask(aid, agent_input, agent_cfg.get('system'))
            tokens = result['total_tokens']
            log(aid, 'COMPLETE', f'{tokens} tokens')
            results[aid] = result['output']
        except Exception as e:
            log(aid, 'FAILURE', str(e))
            raise

        if agent_cfg['approval']['required']:
            if DEMO_AUTO_APPROVE:
                log(aid, 'APPROVAL_GATE', 'auto-approved (DEMO_AUTO_APPROVE=True)')
            else:
                decision = input(f'  Approve {name} output? (y/n): ').strip().lower()
                log(aid, 'APPROVAL_DECISION', decision)
                if decision != 'y':
                    log(aid, 'REJECTED')
                    return results

    print()
    print(SEP)
    print('HARNESS PIPELINE COMPLETE')
    out = results.get('publisher', '')
    if out:
        print()
        print(f'Publisher: {out[:300]}')
    print(SEP)
    return results


print('run_raw() and run_harness() defined.')

## Section 1 — Raw Pipeline

No harness protections. The writer will fail. Watch what happens next.

In [ ]:
run_raw(TOPIC)

**What just happened:**

- The writer failed — but the error was caught and swallowed
- The writer's output was set to an empty string `''`
- The publisher ran anyway — on that empty string
- The researcher got stale input, not the current run's topic output
- No structured log. No timestamps. No audit trail.

The pipeline ran to completion. Nothing looked broken from the outside.

## Section 2 — Harness Pipeline

Same failure. Same pipeline. Autonomee active.

In [ ]:
try:
    run_harness(TOPIC)
except RuntimeError as e:
    print()
    print(f'Pipeline stopped: {e}')

**What just happened:**

- The researcher ran with fresh topic input and scoped tools — logged
- The writer started, failed, and immediately threw — logged with timestamp
- The pipeline stopped. The publisher never ran.
- The structured log shows exactly where, when, and why.

Same failure. Completely different outcome.

---

## Decision 1 — Throw on failure, never swallow

**Failure prevented:** Silent Cascade — Agent 2 stalled, Agent 3 waited indefinitely on empty input.

The pattern difference: `except Exception: return ''` vs `except Exception: raise`

In [ ]:
# Before: failure swallowed — downstream runs on empty output

def step_raw(name, input_text, inject_failure=False):
    if inject_failure:
        try:
            raise RuntimeError(f'{name}: model timeout')
        except Exception as e:
            print(f'  [{name}] error: {e}')
            return ''  # empty — pipeline continues
    return f'(output from {name})'

out_researcher = step_raw('researcher', TOPIC)
out_writer     = step_raw('writer', out_researcher, inject_failure=True)
out_publisher  = step_raw('publisher', out_writer)  # runs on empty string

print()
print(f'writer output:     {repr(out_writer)}')
print(f'publisher received: {repr(out_writer)}')
print(f'publisher ran on empty input: {out_writer == str()}')
print('pipeline stopped at writer failure: No')

In [ ]:
# After: failure propagates — pipeline stops at the first failure

def step_harness(name, input_text, inject_failure=False):
    log(name, 'START')
    if inject_failure:
        log(name, 'FAILURE', 'model timeout')
        raise RuntimeError(f'{name}: model timeout')  # propagates immediately
    log(name, 'COMPLETE')
    return f'(output from {name})'

try:
    out_researcher = step_harness('researcher', TOPIC)
    out_writer     = step_harness('writer', out_researcher, inject_failure=True)
    out_publisher  = step_harness('publisher', out_writer)  # never reached
except RuntimeError as e:
    print()
    print(f'pipeline stopped: {e}')
    print('publisher never ran')
    print('failure is explicit, logged, and propagated')

**Observation:** One `raise` instead of `return ''` is the entire structural difference. The harness enforces it — every agent runner follows the same contract.

---

## Decision 2 — Resolve input in-memory, not from file

**Failure prevented:** Timezone Trap — IST/UTC drift caused agents to read stale files from the prior run.

The pattern difference: `open(file).read()` vs `results[upstream_id]`

In [ ]:
# Before: input read from file — stale content from a prior run
import tempfile, os

# Simulate: a file written by a previous pipeline run
stale_path = os.path.join(tempfile.gettempdir(), 'research.txt')
with open(stale_path, 'w') as f:
    f.write('AI automates tasks. Agents follow instructions.')  # prior run content

def resolve_input_raw(agent_cfg, current_results):
    if agent_cfg['input']['source'] == 'agent':
        with open(stale_path) as f:
            return f.read()  # reads from file — might be from yesterday
    return TOPIC

writer_cfg = CONFIG['agents'][1]

# Researcher just produced fresh output in THIS run:
current_run = {'researcher': '1. Agents need orchestration to be reliable.\n2. Harness controls what agents can do.\n3. Observability catches failures early.'}

writer_input = resolve_input_raw(writer_cfg, current_run)

print(f'Writer received: {writer_input}')
print()
print('Is this from the current run? Unknown — it came from a file.')
print('Could it be stale? Yes.')

In [ ]:
# After: input from in-memory results dict — always from the current run

def resolve_input_harness(agent_cfg, current_results):
    if agent_cfg['input']['source'] == 'agent':
        upstream = agent_cfg['input']['agent_id']
        if upstream not in current_results:
            raise RuntimeError(f'no output from {upstream} in this run')
        return current_results[upstream]  # always from the current run
    return TOPIC

writer_cfg = CONFIG['agents'][1]

# Same current run:
current_run = {'researcher': '1. Agents need orchestration to be reliable.\n2. Harness controls what agents can do.\n3. Observability catches failures early.'}

writer_input = resolve_input_harness(writer_cfg, current_run)

print(f'Writer received: {writer_input}')
print()
print('Is this from the current run? Yes — it came from the results dict.')
print('Could it be stale? No.')

**Observation:** The results dict is populated by the loop on each run. There is no way to read the previous run's output — the dict only exists for the current execution.

---

## Decision 3 — Run on intent, not time

**Failure prevented:** Dumb Executor — CRON job died, downstream agent waited indefinitely for a notification that never came.

The pattern difference: continue-on-failure loop vs stop-and-log-on-failure loop

In [ ]:
# Before: fire-and-forget loop — no intent, no awareness of failure state
print('Raw loop: fire each step regardless')
print()

outputs = []
for agent_cfg in CONFIG['agents']:
    aid = agent_cfg['id']
    try:
        if aid == 'writer':
            raise RuntimeError('writer: model timeout')
        outputs.append(f'{aid}: ran')
    except Exception:
        outputs.append(f'{aid}: failed (swallowed)')  # loop does not stop

for o in outputs:
    print(f'  {o}')

ran = [o for o in outputs if 'ran' in o]
failed = [o for o in outputs if 'failed' in o]
print()
print(f'Steps ran: {len(ran)}  Failed: {len(failed)} (silently)')
print('Loop knew what to do when writer failed? No.')
print('Publisher ran after writer failure? Yes.')

In [ ]:
# After: intent-aware loop — stops cleanly on failure, logs the cause
print('Harness loop: intent-aware')
print()

results = {}
stopped_at = None

for agent_cfg in CONFIG['agents']:
    aid = agent_cfg['id']
    log(aid, 'START')
    try:
        if aid == 'writer':
            raise RuntimeError('writer: model timeout')  # same failure
        results[aid] = f'{aid}: completed'
        log(aid, 'COMPLETE')
    except RuntimeError as e:
        log(aid, 'FAILURE', str(e))
        stopped_at = aid
        break  # explicit intent: stop here, do not run downstream agents

print()
if stopped_at:
    not_run = [a['id'] for a in CONFIG['agents'] if a['id'] not in results]
    print(f'Pipeline stopped at: {stopped_at}')
    print(f'Agents that did not run: {not_run}')
else:
    print('Pipeline completed.')

**Observation:** The `break` is the intent. A CRON job has no equivalent — it fires at a time and exits. The harness loop knows which step it is on, what it received, and decides what to do next.

---

## Decision 4 — Scope tools per agent from config

**Failure prevented:** Wrong Tool — agent called the language model directly instead of the provided script. ~6× more expensive. Invisible until a manual trace audit.

The pattern difference: `all_tools` passed to every agent vs `filtered_tools` from agent config

In [ ]:
# Before: all tools available to all agents
researcher_cfg = CONFIG['agents'][0]  # supposed to use only 'search'
allowed = researcher_cfg['tools']

print(f'Researcher config declares: {allowed}')
print(f'Tools actually available:   {list(ALL_TOOLS.keys())}')
print()

# Model has access to 'direct_llm' — a tool it should never call
wrong_tool = 'direct_llm'
if wrong_tool in ALL_TOOLS:
    result = ALL_TOOLS[wrong_tool]('research AI agent failures in production')
    print(f'Researcher called {wrong_tool!r}: {result}')
    print()
    print('Authorized? No.  Blocked? No.  Logged? No — not until manual trace audit.')

In [ ]:
# After: tool scope enforced at registration — model never sees unauthorized tools

def scope_tools(agent_cfg, all_tools):
    allowed = set(agent_cfg['tools'])
    scoped = {k: v for k, v in all_tools.items() if k in allowed}
    log(agent_cfg['id'], 'TOOLS_REGISTERED', str(sorted(allowed)))
    return scoped

researcher_cfg = CONFIG['agents'][0]
scoped = scope_tools(researcher_cfg, ALL_TOOLS)

print(f'Researcher tools after scoping: {list(scoped.keys())}')
print()

wrong_tool = 'direct_llm'
if wrong_tool in scoped:
    print(f'called {wrong_tool}: {scoped[wrong_tool]("...")}')
else:
    print(f'{wrong_tool!r} not in scope — researcher cannot call it.')
    print('The model never sees it as an option.')
    print()
    result = scoped['search']('AI agent failures in production')
    print(f'Correct tool used: {result}')

**Observation:** Scoping happens at registration time — before the agent runs. The model is never offered a tool it should not use. There is nothing to audit after the fact because the wrong call is structurally impossible.

In [ ]:
# Summary: raw vs harness across all four decisions

rows = [
    ('Failure propagation', 'swallowed — pipeline continues',  'throws — pipeline stops cleanly'),
    ('Input resolution',    'from file — may be stale',        'from results dict — always fresh'),
    ('Loop intent',         'continues on failure',            'stops and logs on failure'),
    ('Tool scope',          'all tools available',             'only config-declared tools'),
]

col1, col2, col3 = 26, 36, 36
header = f'{"Decision":<{col1}} {"Raw (no harness)":<{col2}} {"Harness"}'
print(header)
print('-' * (col1 + col2 + col3))
for decision, raw, harness in rows:
    print(f'{decision:<{col1}} {raw:<{col2}} {harness}')

print()
print('The harness does not make the model smarter.')
print('It makes the system predictable, observable, and controllable.')

In [ ]:
# ── Try your own topic ────────────────────────────────────────────────────────
# Replace MY_TOPIC and run the full harness pipeline end to end.
# ─────────────────────────────────────────────────────────────────────────────
MY_TOPIC = 'the future of multi-agent AI systems'  # ← replace with your topic

SIMULATE_FAILURE = False  # full run — no injected failure

print(f'Running Autonomee harness on: {MY_TOPIC!r}')
print()
run_harness(MY_TOPIC)

## Next Steps

- **Part 1:** [AI Agent Scaffolding: Four Failures and What Prevents Them](https://sriharshacr.github.io/blogs/ai-agent-scaffolding-four-failures/) — the real failures that motivated these four decisions
- **Repo:** [github.com/SriharshaCR/autonomee](https://github.com/SriharshaCR/autonomee) — the full implementation

---

*Part of the Agentic Harness — AI's Operating System series*